# Notebook 11 — Scaling Laws and Training-Compute Planning

    ## Learning objectives

    - Estimate parameter, token, memory, and FLOP budgets
- Fit cautious empirical scaling curves
- Design pilot runs and stopping decisions

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 11.1 What scales

Model size, active parameters, training tokens, sequence length, batch tokens, optimizer updates, numerical precision, and hardware utilization are different quantities. Dense decoder training FLOPs are often estimated with a constant times parameters times tokens, but embeddings, attention, MoE routing, rematerialization, and inefficient kernels change the constant. Report assumptions and measured throughput. Epochs are misleading across corpora; tokens and compute make experiments comparable. Storage also includes raw/tokenized data, optimizer state, checkpoints, logs, and evaluation artifacts.


In [ ]:
def dense_flops(params,tokens,multiplier=6): return multiplier*params*tokens
for p,t in [(1e8,2e9),(1e9,2e10),(7e9,1.4e11)]: print(p,t,dense_flops(p,t))


## 11.2 Empirical power laws

Loss frequently follows approximate power laws over useful scale ranges, with an irreducible floor. Fit in log space only after plotting residuals and separating architecture or data-regime changes. A smooth aggregate curve can hide language, domain, memorization, safety, or downstream regressions. Extrapolation beyond observed compute has wide structural uncertainty. Scaling laws help allocate pilots and detect underperforming runs; they do not replace evaluation or guarantee emergent behavior.


In [ ]:
import numpy as np
compute=np.array([1,2,4,8,16.]); loss=2.1+.8*compute**-.3
coef=np.polyfit(np.log(compute),np.log(loss-2.1),1); print("exponent",coef[0])


## 11.3 Compute-optimal tradeoffs

Given fixed compute, increasing parameters leaves fewer tokens and increasing tokens leaves a smaller model. Compute-optimal guidance depends on data quality, reuse, inference demand, and target capabilities. A smaller model trained longer can be preferable when serving dominates lifetime cost; a larger undertrained model may adapt differently. Deduplicated unique tokens, repeated tokens, synthetic mixtures, and curriculum order are not interchangeable. Treat token-to-parameter ratios as hypotheses to test with pilots, not universal constants.


In [ ]:
budget=6e20
for params in [1e8,5e8,1e9,3e9]: print(params,"tokens",budget/(6*params))


## 11.4 Planning and monitoring

Run small matched pilots varying one scale axis, fit curves with uncertainty, and reserve budget for failures, evaluation, checkpointing, and ablations. Estimate memory for weights, gradients, optimizer states, activations, KV/cache workspaces, and fragmentation. During training compare observed loss with predicted bands, tokens per second, utilization, gradient norms, and data health. Stop or intervene on evidence, not sunk cost. Archive failed runs and document carbon, financial, and opportunity costs alongside quality.


In [ ]:
def gib(params,weight=2,grad=2,optimizer=8): return params*(weight+grad+optimizer)/2**30
for p in [1e8,1e9,7e9]: print(p,gib(p))


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## Exercises

    1. Fit a power law and bootstrap its exponent.
2. Plan three matched pilot runs.
3. Compare training and lifetime inference compute.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
